# lets Just Look at Astroquery #

In [1]:
from astroquery.ipac.ned import Ned
import numpy as np

# Set numpy print options for full display
np.set_printoptions(threshold=np.inf, linewidth=1000)

# Query
sn_name = "2008050"
result_table = Ned.query_object(sn_name)

# Print full table
print(result_table)

# Also print column names and values explicitly
print("\n--- Full Column Breakdown ---")
for col in result_table.colnames:
    print(f"{col}: {result_table[col][0]}")

RemoteServiceError: The remote service returned the following error message.
ERROR: The object name that you have submitted is not currently recognized by the NED name interpreter.

In [2]:
Ned.query_object()

TypeError: NedClass.query_object() missing 1 required positional argument: 'object_name'

# Take the table with SNID's and replace the SNID Collumn with "SN" + SNID for propper input in astroquery #

In [ ]:
import pandas as pd

# Load the CSV
df = pd.read_csv("DES5YR_DESI_match_data/snia_with_missing_host_coords.csv")

# Prepend "SN" to each SNID
df["SNID"] = "SN" + df["SNID"].astype(str)

# Save it back
df.to_csv("DES5YR_DESI_match_data/snia_with_missing_host_coords_snid_4_astroquery.csv", index=False)

# Check how many of the Modified SNIDS return anything useful from Astroquery #

In [ ]:
import pandas as pd
from astroquery.simbad import Simbad
import time

# Load the CSV with preformatted SNIDs
df = pd.read_csv("DES5YR_DESI_match_data/snia_with_missing_host_coords_snid_4_astroquery.csv")

# SIMBAD setup
simbad = Simbad()
simbad.TIMEOUT = 10

# Function to check if SIMBAD finds the object
def check_simbad(sn_name):
    try:
        return simbad.query_object(sn_name) is not None
    except:
        return False

# Build result list
results = []
for sn in df['SNID']:
    found = check_simbad(sn)
    results.append({'SN': sn, 'Found': found})
    time.sleep(0.5)  # polite delay

# Save the SN,Found table
out_df = pd.DataFrame(results)
out_df.to_csv("DES5YR_DESI_match_data/astroquery_found_table.csv", index=False)

print("✅ Done. Saved: astroquery_found_table.csv")
print(len(out_df))

# Copy the RA, DEC, and REDSHIFT of the SNIa to Astroquery  #

In [ ]:
import pandas as pd
from astroquery.ipac.ned import Ned
import time

# Load your SN list
df = pd.read_csv("DES5YR_DESI_match_data/snia_with_missing_host_coords_snid_4_astroquery.csv")

# Initialize result columns
df["RA"] = None
df["DEC"] = None
df["Redshift"] = None

# Query NED and append results
for i, sn in enumerate(df["SNID"]):
    try:
        res = Ned.query_object(sn)
        df.at[i, "RA"] = res["RA"][0]
        df.at[i, "DEC"] = res["DEC"][0]
        df.at[i, "Redshift"] = res["Redshift"][0]
    except Exception as e:
        print(f"⚠️ Failed for {sn}: {e}")
    time.sleep(0.5)  # polite pause

# Save output
df.to_csv("DES5YR_DESI_match_data/snia_with_coords_and_redshift.csv", index=False)
print("✅ Done. Saved: snia_with_coords_and_redshift.csv")